# Week 2

### Goals
- Understand system and user instructions.
- Turn free-form responses into a predictable format.

### Tasks
- Try several prompts for extracting a movie request: genre, mood, maximum runtime and exclusions.
- Ask for structured JSON and validate the fields with Pydantic or simple Python checks.
- Handle missing or ambiguous preferences explicitly.
- Compare two prompt variants on five example requests.
- Separate the model’s interpretation from catalog facts.

### Expected Output
- 02_prompting.ipynb with a validated preference object.
- Five example requests and a brief comparison of failures.


---
#### Simple Prompt with JSON format response

In [19]:
from ollama import chat

response = chat(
  model='llama3.2',
  messages=[{'role': 'user', 'content': 'Tell me about Switzerland.'}],
  format='json'
)
print(response.message.content)

{"Switzerland":{"name":"Switzerland","description":"Switzerland","type":"Country"}}


**Observations:**  
Prompt seems to be very unstable. Often the response is just an empty JSON, sometimes it just some minimal "information" is given, looking something like this:
{"Switzerland":{"name":"Switzerland","description":"Switzerland","type":"Country"}}

Only very few times an actual text response answering the question is returned.

My Assumption is that this is due to the JSON format just telling the model to return its answer in this format, not what the JSON should look like. Therefore for small models such as the llama3.2 it just "guesses" a response. In its eyes the empty JSON response fullfills the return format constraint, because i did not strictly tell it waht it should be filled with. Therefore sometimes it is empty, sometimes it returns minimal answers to the actual question, sometimes a full text answer.

With Pydantic i define exactly those missing constraints. I can definde exactly which information or "keys" the model should return for my initial text request and in what format and datatype (e.g. string).

#### Same user request with Pydantic constraints

In [21]:
from pydantic import BaseModel

class Country(BaseModel):
  name: str
  capital: str
  languages: list[str]
  currency: str
  population: int

response = chat(
  model='llama3.2',
  messages=[{'role': 'user', 'content': 'Tell me about Switzerland.'}],
  format=Country.model_json_schema(),
)

country = Country.model_validate_json(response.message.content)
print(country)

name='Switzerland' capital='Bern' languages=['German', 'French', 'Italian', 'Romansh'] currency='$' population=8592811


**Observations:**  
Response is much more stable, i receive clear, simple responses for each of the keywords listed in the Country Class, although those answers are not always 100% correct or exactly the same for each run (probably a limit of the small model).

--- 
#### **Extract movie requests**

**Output structure**

In [36]:
# Define the structure for a movie request
class MovieRequest(BaseModel):
    genre: str | None               # Add None to indicate that this can also be empty if not specified by the user
    mood: str | None
    maximum_runtime: int | None
    exclusions: list[str] | None

In [37]:
test_requests = [
    'I want to watch a scary horror movie that is not longer than 90 minutes and not about zombies.',
    'I would like to watch a short, romantic comedy.',
    'I want to watch a fast paced action movie, but not about superheroes.',
    'Historical drama that is not about any wars and not longer than 2 hours.',
    'I want to watch a good movie.'
]

##### **Without system prompt**

In [ ]:
for user_input in test_requests:
    print(f'Input: {user_input}')
    
    response = chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': user_input}],
        format=MovieRequest.model_json_schema()
    )
    
    request = MovieRequest.model_validate_json(response.message.content)
    print(f'Extracted: {request}\n' + '-'*40)

Input: I want to watch a scary horror movie that is not longer than 90 minutes and not about zombies.
Extracted: genre='Horror' mood='Frightening' maximum_runtime=90 exclusions=['Zombie', 'Monster']
----------------------------------------
Input: I would like to watch a short, romantic comedy.
Extracted: genre='romantic comedy' mood='light-hearted' maximum_runtime=90 exclusions=['dark themes', 'disturbing content']
----------------------------------------
Input: I want to watch a fast paced action movie, but not about superheroes.
Extracted: genre='   Action: Explosive, high-energy films that focus on intense physical action and thrills, often with a strong emphasis on suspense and excitement. Examples include ' mood='   Fast-paced, adrenaline-fueled, and action-packed, with a focus on intense sequences and explosive set pieces. Examples include ' maximum_runtime=90 exclusions=['Expendables (2010)', 'The Accountant (2016)', 'Atomic Blonde (2017)', 'Mission: Impossible - Fallout (2018)'

##### **WITH system prompt**

In [39]:
for user_input in test_requests:
    print(f'Input: {user_input}')
    
    response = chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': 'You are a preference extraction assistant. You will receive a user request for a movie recommendation. Your taks is to extract the preferences in a structured JSON format. Extract only what is explicitly mentioned by the user. Do not invent, guess or add any extra information.'},
            {'role': 'user', 'content': user_input}
        ],
        format=MovieRequest.model_json_schema()
    )
    
    request = MovieRequest.model_validate_json(response.message.content)
    print(f'Extracted: {request}\n' + '-'*40)

Input: I want to watch a scary horror movie that is not longer than 90 minutes and not about zombies.
Extracted: genre='Horror' mood='Scary' maximum_runtime=90 exclusions=['Zombies']
----------------------------------------
Input: I would like to watch a short, romantic comedy.
Extracted: genre='romantic comedy' mood='short' maximum_runtime=0 exclusions=[]
----------------------------------------
Input: I want to watch a fast paced action movie, but not about superheroes.
Extracted: genre='action' mood='fast-paced' maximum_runtime=None exclusions=['superhero']
----------------------------------------
Input: Historical drama that is not about any wars and not longer than 2 hours.
Extracted: genre='Historical Drama' mood=None maximum_runtime=120 exclusions=['war']
----------------------------------------
Input: I want to watch a good movie.
Extracted: genre=None mood=None maximum_runtime=None exclusions=None
----------------------------------------


#### **Observations**

Generally, with the system prompt telling the model exactly what its job is and how it needs to behave, helps to get more consistant and more accurate responses. Without the system prompt, the model functions, but it tends to invent information that was not provided, especially when the user requests are vague. For example when the user only tells mentions a "good movie", so no clear preferences, without a system prompt the model invents or assumes what the user means with a "good" movie: `genre='Action' mood='Adrenaline-fueled' maximum_runtime=136 exclusions=['romance', 'comedy']`. but this is of course different for every person. For the same exact user prompt, but added system prompt, the model correctly returns `None` for the respective keys, since no clarification was given by the user.  
Second example: FOr teh first prompt asking for a horror movie without zombies (very specific and clear preferences) the simple model without a system prompt invents a exclusion "horror movies", although the user specifically asks for that. FOr other runs sometimes also "vampires" was added in this list. So the model just tends to add additional similar information/topics that it associates with the user input.

The system promt also doesn't work perfectly (though better and more consistant than without one): The different Keys in the JSON format may need to be specified more in the system prompt, so the model clearly knows what to associate with what. E.g. For a "short  romantic comedy", the model specified the mood as short and max_runtime as 0, which obviously is wrong.

**Conclusion:**  
With a system prompt the model already performs generally better and more consistant than simply giving the model the user message without a system prompt. The accuracy of the request extraction can most likely be improved further by using a more detailed system prompt explaining its task and the exact JSON keys (or fields) more accuratly.